# W0b: Optional Bridge to a First Tested Calculation

This optional bridge uses the working toolchain for a small engineering calculation. It previews function contracts, validation, and tests; those ideas are taught systematically in Week 2, so you may skip this notebook during setup.

> __Learning Objectives:__
>
> By the end of this optional activity, you should be able to:
> * __State a unit contract:__ Translate the ideal-gas equation into a function that declares the units and the admissible range of every argument. A formula on its own says nothing about what its symbols mean or which values make sense.
> * __Reject inadmissible inputs:__ Raise an error naming the offending argument when a value is zero, negative, or not finite. Failing at the boundary is cheaper than returning a number that is quietly wrong.
> * __Check a result against a known case:__ Use a reference calculation and an executable test to confirm that the implementation does what the contract promises. Those tests hold the behavior in place as the implementation changes.

Let's get started!
___


## Setup, Data, and Prerequisites

Complete W0a first. This notebook uses the same pinned course environment and loads the student-facing function stub from `src/Compute.jl`.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, and includes the local source in `src/`. The second line guards against the case where the notebook front end started in a different directory and resolved a different setup file. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:


In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # activate the environment and load local source
isdefined(Main, :CHEME5800_W0B_ROOT) ||
    error("Setup resolved the wrong Include.jl. Open this notebook from inside its own W0b folder, then restart the kernel.");

The setup cell completed, so the pinned environment, [the `Test` standard library](https://docs.julialang.org/en/v1/stdlib/Test/), and [`src/Compute.jl`](src/Compute.jl) are loaded. That last file holds the function you are about to write. What follows is a single calculation carried through in full: state the contract, implement it, and check the result against a case whose answer is already known.

___


## A first tested calculation

The toolchain works. Now use it for something. An engineering calculation is a formula plus a statement of what its symbols mean and which values are admissible. Writing that contract down first is what turns a formula into something a test can check. The [ideal gas law](https://en.wikipedia.org/wiki/Ideal_gas_law) relates the state variables of a gas:

$$
PV = nRT
$$

We want pressure, so rearrange for $P$:

$$
P = \frac{nRT}{V}
$$

> __The contract:__
>
> * $n$ is the amount of substance, in $\mathrm{mol}$. It must be finite and strictly positive.
> * $T$ is the absolute temperature, in $\mathrm{K}$. It must be finite and strictly positive, because at or below absolute zero the model does not apply.
> * $V$ is the volume, in $\mathrm{m^{3}}$. It must be finite and strictly positive.
> * $R$ is the [universal gas constant](https://en.wikipedia.org/wiki/Gas_constant), $8.31446261815324\;\mathrm{Pa\,m^{3}\,mol^{-1}\,K^{-1}}$, exact by definition since the 2019 SI redefinition. It is a keyword argument whose default is that value, so you will not normally pass it, but the function validates it along with the other three.
> * The return value $P$ is the pressure, in $\mathrm{Pa}$.

[The `ideal_gas_pressure(...)` function in `src/Compute.jl`](src/Compute.jl) currently holds a signature, a docstring, and two `TODO` comments.

## Implement the function

> __What to write:__
>
> * __Validate the inputs.__ Throw an [`ArgumentError`](https://docs.julialang.org/en/v1/base/base/#Core.ArgumentError) naming the argument when any of them is not finite, or not strictly positive. [The `isfinite(...)` function](https://docs.julialang.org/en/v1/base/numbers/#Base.isfinite) does the first check.
> * __Return the pressure.__ Compute $P = nRT/V$ and return it as a `Float64`.

Open the file, complete both `TODO`s, then restart the kernel and run this notebook from the top. The restart is required: `Include.jl` skips the local source once `Week00Bridge` is loaded, so an edit to `src/Compute.jl` has no effect until the kernel restarts. Until both `TODO`s are done, the next code cell stops with a "not implemented yet" error, which is the expected starting state.

## Check against a case you already know

A new function deserves a case whose answer you already know. One mole at $273.15\;\mathrm{K}$ in about $22.414\;\mathrm{L}$ should come back near one atmosphere.

> __Where does $22.414\;\mathrm{L}$ come from?__
>
> It is not a measurement, it is a rearrangement: $V = nRT/P$ with $n = 1$, $T = 273.15\;\mathrm{K}$, and $P = 101325\;\mathrm{Pa}$. Feeding that volume back in should therefore return one atmosphere. This confirms the arithmetic inverts correctly; it is not independent evidence about any real gas.

We store the result in `pressure_Pa::Float64` and its kilopascal equivalent in `pressure_kPa::Float64`:

In [ ]:
pressure_Pa, pressure_kPa = let

    # initialize -
    amount_mol = 1.0; # amount of substance, mol
    temperature_K = 273.15; # absolute temperature, K
    volume_m3 = 0.02241396954; # volume, m^3

    # compute -
    P = ideal_gas_pressure(amount_mol, temperature_K, volume_m3);

    P, P/1000 # return
end
(pressure_Pa = pressure_Pa, pressure_kPa = pressure_kPa)

Once both `TODO`s are complete, the cell above reports about $101.325\;\mathrm{kPa}$, or one atmosphere, which is the number worth carrying in your head as a sanity check for gas-phase work.

___

## Tests

After completing both `TODO`s in `src/Compute.jl`, restart the kernel, run from the top, and use these tests as the executable version of the function contract.


In [ ]:
let
    @testset verbose = true "CHEME 4/5800 Week 0 Optional Bridge" begin
        @test isapprox(pressure_Pa, 101_325.0; rtol = 1e-8)
        @test ideal_gas_pressure(2, 300, 0.05) isa Float64
        @test_throws ArgumentError ideal_gas_pressure(0, 300, 0.05)
        @test_throws ArgumentError ideal_gas_pressure(1, -10, 0.05)
        @test_throws ArgumentError ideal_gas_pressure(1, 300, Inf)
        @test_throws ArgumentError ideal_gas_pressure(1, 300, 0.05; gas_constant = -1)
    end
end;


___


## Summary

An engineering calculation is more than a formula: its interface must state units, admissible inputs, and expected output.

> __Key Takeaways:__
>
> * **A contract makes a formula checkable:** Naming the units and the admissible range of every argument turns a bare formula into a statement a test can hold to account. Without that statement there is nothing for a test to verify.
> * **Reject bad input at the boundary:** An error that names the offending argument stops a physically meaningless call before it produces a number. A wrong answer that looks plausible costs more than an error that arrives immediately.
> * **A reference case checks arithmetic, not physics:** Recovering one atmosphere from the molar volume confirms that the implementation inverts correctly, and nothing more. Tests then preserve that behavior as the implementation changes.

Week 2 develops these ideas in the required functions, errors, testing, and debugging sequence.
___